In [ ]:
# Requisitos: pip install numpy matplotlib rainflow
import numpy as np
import rainflow                      # python-rainflow (ASTM E1049, 3 puntos)
import matplotlib
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# 1. Historiales de esfuerzo en la muesca (MPa): sigma = Kf*32*M/(pi d^3), tau = Kfs*16*T/(pi d^3)
# ------------------------------------------------------------------
M = np.array([0, 340, -260, 420, -30, 330, -400, 200, -200, 0.])      # N·m
T = np.array([0, 200, -200, 100, -300, 400, -100, 150, -50, 0.])      # N·m
Kf, Kfs, d = 1.539, 1.369, 30e-3                                      # d en m
sigma_x = Kf  * 32 * M / (np.pi * d**3) / 1e6                         # MPa (197, -151, 244, ...)
tau_xy  = Kfs * 16 * T / (np.pi * d**3) / 1e6                         # MPa (52, -52, 26, -77.5, ...)

# ------------------------------------------------------------------
# 2. Constantes del material (se calculan, no se escriben a mano)
# ------------------------------------------------------------------
Sut, f_res, Se = 460.0, 0.9, 142.18            # MPa (Se de la hoja de cálculo)
k = 0.3                                        # constante de Findley (aceros)
a = (f_res * Sut) ** 2 / Se                    # Basquin: sigma = a N^b
b = -np.log10(f_res * Sut / Se) / 3.0          # = -0.1547 (sin redondear)
raiz = np.sqrt(1 + k**2)
fe = 0.5 * Se * (k + raiz)                     # limite de Findley (calibrado con ensayo UNIAXIAL a Se) = 95.5 MPa
aF = 0.5 * a * (k + raiz)                      # Basquin ajustado a Findley: f = aF N^b  -> 810 MPa

# ------------------------------------------------------------------
# 3. Barrido de planos
# ------------------------------------------------------------------
thetas_deg = np.arange(0, 181, 1)
dano_total = np.zeros(len(thetas_deg))         # Miner + rainflow (riguroso)
f_envolvente = np.zeros(len(thetas_deg))       # ciclo unico: dTau_max/2 + k*sigma_n,max global (metodo manual)

def esfuerzos_plano(theta):
    sn = (sigma_x / 2) * (1 + np.cos(2 * theta)) + tau_xy * np.sin(2 * theta)
    ta = -(sigma_x / 2) * np.sin(2 * theta) + tau_xy * np.cos(2 * theta)
    return sn, ta

for idx, theta in enumerate(np.radians(thetas_deg)):
    sn, ta = esfuerzos_plano(theta)
    f_envolvente[idx] = (ta.max() - ta.min()) / 2 + k * sn.max()
    dano = 0.0
    for rng, media, conteo, i_ini, i_fin in rainflow.extract_cycles(ta):
        lo, hi = min(i_ini, i_fin), max(i_ini, i_fin)
        f_ciclo = rng / 2 + k * sn[lo:hi + 1].max()       # sigma_n,max concurrente con el ciclo
        if f_ciclo > fe:                                   # por debajo de fe: sin daño
            dano += conteo / (f_ciclo / aF) ** (1 / b)
    dano_total[idx] = dano

i_crit = np.argmax(dano_total)
theta_crit, dano_max = thetas_deg[i_crit], dano_total[i_crit]
vida_miner = 1 / dano_max

# Método manual simplificado (un solo ciclo crítico por bloque), evaluado en el MISMO plano crítico
f_max = f_envolvente[i_crit]
vida_manual = (f_max / aF) ** (1 / b)

print("-" * 60)
print(f"Constantes: a = {a:.2f} MPa, b = {b:.4f}, fe = {fe:.2f} MPa, aF = {aF:.1f} MPa")
print(f"[Rainflow + Miner] plano critico = {theta_crit}°, dano/bloque = {dano_max:.3e}, vida = {vida_miner:,.0f} bloques")
print(f"[Ciclo unico (manual)] plano = {theta_crit}°, f = {f_max:.1f} MPa, n = fe/f = {fe/f_max:.2f}, vida = {vida_manual:,.0f} ciclos")
print("-" * 60)

# ------------------------------------------------------------------
# 4. Tabla de ciclos en el plano critico (para contrastar con la hoja manual)
# ------------------------------------------------------------------
th = np.radians(theta_crit)
sn, ta = esfuerzos_plano(th)
print(f"Ciclos de rainflow en theta = {theta_crit}°")
for rng, media, conteo, i_ini, i_fin in rainflow.extract_cycles(ta):
    lo, hi = min(i_ini, i_fin), max(i_ini, i_fin)
    f_c = rng / 2 + k * sn[lo:hi + 1].max()
    dc = conteo / (f_c / aF) ** (1 / b) if f_c > fe else 0.0
    print(f"  puntos {lo}-{hi}: rango = {rng:6.1f}  n = {conteo}  sn_max = {sn[lo:hi+1].max():6.1f}  f = {f_c:6.1f}  D = {dc:.3e}")

# ------------------------------------------------------------------
# 5. Grafica
# ------------------------------------------------------------------
plt.figure(figsize=(10, 5))
plt.plot(thetas_deg, dano_total, label="Daño de Palmgren-Miner", color="purple", linewidth=2)
plt.axvline(x=theta_crit, color="red", linestyle="--", label=f"Plano crítico ({theta_crit}°)")
plt.plot(theta_crit, dano_max, "ro", markersize=8)
plt.title("Identificación del Plano Crítico mediante Daño Acumulado (Rainflow)", fontsize=12)
plt.xlabel(r"Ángulo del plano $\theta$ (grados)", fontsize=11)   # r"..." evita que \t se lea como tabulador
plt.ylabel("Daño total por bloque (1/N)", fontsize=11)
plt.xticks(np.arange(0, 181, 15)); plt.xlim(0, 180)
plt.grid(True, linestyle=":", alpha=0.8); plt.legend(); plt.tight_layout()
plt.savefig("fig_P3_plano_critico.png", dpi=300)
plt.show()
